# Two-Way Swaps, One Run Command
**Choose a benchmark. Choose a harness. Run.**

Compare **Hermes → Pi** on the same SWE-Pro task, then keep Pi and switch to **Terminal-Bench 2.1**.
The task runs in a sandbox and gets a real benchmark reward.

*Preview branch. One task per pair demonstrates the workflow, not benchmark accuracy.*
<!-- SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved. -->
<!-- SPDX-License-Identifier: Apache-2.0 -->

## Setup — before recording
Follow the sibling README to install Gym and prepare data on a Linux Docker host.
The kernel must use this checkout's `.venv`. The endpoint and key stay outside saved cells.

In [ ]:
import getpass
import os
from pathlib import Path
import subprocess
import sys
import signal
from IPython.display import clear_output

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'nemo_gym').is_dir())
os.chdir(ROOT)
for name, prompt in [
    ('DEMO_MODEL_URL', 'Chat Completions base URL (ending /v1): '),
    ('DEMO_MODEL_NAME', 'Model name: '),
    ('DEMO_GYM_HOST', 'This host address, reachable from Docker: '),
]:
    if not os.environ.get(name):
        os.environ[name] = input(prompt).strip()
if not os.environ.get('DEMO_MODEL_KEY'):
    os.environ['DEMO_MODEL_KEY'] = getpass.getpass('Model API key (not saved): ')
print('Setup ready. Credentials are not printed or saved in the notebook.')

In [ ]:
# Use the same selected SWE task file for Hermes and Pi.
INPUTS = {
    'swe-pro': Path('examples/harness-swaps/data/swe-pro.jsonl'),
    'tb21': Path('examples/harness-swaps/data/tb21.jsonl'),
}
TASKS_PER_RUN = 1  # Change to 2 only when both files contain at least two selected tasks.
for name, path in INPUTS.items():
    assert path.is_file(), f'Prepare {name} data or edit INPUTS: {path}'

def run(*, harness: str, benchmark: str) -> None:
    """Thin notebook display wrapper; the helper invokes the normal Gym CLI."""
    command = [sys.executable, '-u', 'examples/harness-swaps/run.py',
               '--harness', harness, '--benchmark', benchmark,
               '--input', str(INPUTS[benchmark]), '--limit', str(TASKS_PER_RUN)]
    with subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, errors='replace') as process:
        try:
            visible = []
            for line in process.stdout:
                visible.append(line.rstrip())
                clear_output(wait=True)
                print('\n'.join(visible[-8:]), flush=True)
            if process.wait():
                raise RuntimeError('Run incomplete. Inspect the saved logs before continuing.')
        except KeyboardInterrupt:
            process.send_signal(signal.SIGINT)
            process.wait(timeout=180)
            raise
print('Ready for the demo. Each run keeps its own result folder.')

## 1 · Start with Hermes × SWE-Pro
A real code task, a sandbox, and the benchmark's own tests.

In [ ]:
run(harness="hermes", benchmark="swe-pro")

## 2 · Swap the harness
**Change only `harness`.** Keep the same task, model and verifier.

In [ ]:
run(harness="pi", benchmark="swe-pro")

## 3 · Swap the benchmark
**Change only `benchmark`.** Keep Pi and the same model.

In [ ]:
run(harness="pi", benchmark="tb21")

## What changed?

| Step | Harness | Benchmark | Integration code changed |
|---|---|---|---|
| Start | Hermes | SWE-Pro | None |
| Swap harness | **Pi** | SWE-Pro | None |
| Swap benchmark | Pi | **TB2.1** | None |

These are compatible integrations in this preview branch. A successful one-task
run is not evidence that every pairing or every benchmark task works.

---
## How it works — optional, after the demo

Users select a harness and benchmark; the demo composes their independent configs.
The shared EnvironmentServer handles setup, execution, verification and cleanup.
`run.py` invokes `gym env start` and `gym eval run --no-serve` and records the exact commands.
Both harnesses execute **inside the task sandbox**.

**Add a harness once:** implement its Agent adapter and runtime/tool mapping.
**Add a benchmark once:** prepare tasks and implement Resources setup, verification and cleanup.
Other compatible counterparts need configuration and smoke validation—not a new pair-specific adapter.

No invented before/after LOC savings: some older pairs were already config-only.
The improvement is independent reusable contracts. See README for source pins and limits.